# Few-shot classification (Methods §3.3)

Each conversation is sent to `gpt-4o-mini` (temperature 0, timeout 1,500 s) with the three prompts of
the coding scheme; the model splits every message into sentences and labels them. The response is
converted into a dataframe, the labels are checked against the scheme, and the first-level category
(`type`) of each label is added.

* **Input:** `df_chats.csv`, the prompts and `tags.json` in `coding_schemes/`
* **Output:** `labeled_conversations/<coding scheme>.csv`
* **Check:** no label outside the coding scheme survives (counted in `02_out_of_scheme_labels`)

**Running the classification again costs money and needs an API key.** With `RUN_API = False` (the
default) the notebook reads the labels obtained in the study from `source/llm_labels_<scheme>.csv` and
only runs the checks. Model output changes over time, so a new run will not reproduce them exactly.

In [2]:
import pandas as pd
from pathlib import Path
import os

# Dataset to work on: 'data' is the study data, 'data_example' the synthetic example.
DATA = Path('..') / os.environ.get('DATASET_FOLDER', 'data')

df = pd.read_csv(DATA / 'df_chats.csv')

RUN_API = False  # True re-runs the classification against the OpenAI API

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1431 entries, 0 to 1430
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   activity_id   1431 non-null   object
 1   user_id       1431 non-null   object
 2   group         1431 non-null   int64 
 3   role          1431 non-null   object
 4   message_num   1431 non-null   int64 
 5   sentence_num  1431 non-null   int64 
 6   sentence_en   1431 non-null   object
dtypes: int64(3), object(4)
memory usage: 78.4+ KB


## Group utterances into one single conversation

In [4]:
df['message_num'] = df['message_num'].astype(int)  # Ensure turns are integers for sorting
grouped = df.groupby(['activity_id', 'user_id'], sort=False)

conversations_json = []
for activity_id, group in grouped:
    conversation = group.sort_values('message_num').apply(
        lambda row: {"message_num": row['message_num'], "role": row['role'], "sentence_en": row['sentence_en']},
        axis=1
    ).tolist()
    conversations_json.append({
        "activity_id": activity_id[0],
        "user_id": activity_id[1],
        "conversation": conversation
    })

len(conversations_json)

159

## Functions for prompt reading, classify sentences and conversations

In [5]:
from openai import OpenAI
from tqdm import tqdm


def read_prompts(folder: str):

    path = Path('..') / "coding_schemes" / folder

    PROMPT_SYSTEM_NAME = path / 'prompt_system.txt'
    PROMPT_USER_NAME = path / 'prompt_user.txt'
    PROMPT_AGENT_NAME = path / 'prompt_agent.txt'

    with open(PROMPT_SYSTEM_NAME) as f:
        PROMPT_SYSTEM = f.read()

    with open(PROMPT_USER_NAME) as f:
        PROMPT_USER = f.read()

    with open(PROMPT_AGENT_NAME) as f:
        PROMPT_AGENT = f.read()

    return (PROMPT_SYSTEM, PROMPT_USER, PROMPT_AGENT)

errors = []
def classify_sentences(client: OpenAI,
                       message: str,
                       prompt_system: str,
                       prompt_user: str,
                       prompt_agent: str):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": prompt_system},
            {"role": "user", "content": prompt_user},
            {"role": "assistant", "content": prompt_agent},
            {"role": "user", "content": message}
        ],
        temperature=0,
        stream=False,
        timeout=1500  # Set the timeout to 1500 seconds
    )

    # handle errors
    try:
        return response.choices[0].message.content
    except:
        errors.append({'message': message, 'response': response})
        return None


def classify_conversation(openai_client: OpenAI,
                          prompt_system: str,
                          prompt_user: str,
                          prompt_agent: str,
                          conversations: list):
    out = []
    for conversation in tqdm(conversations, desc='Classifying...'):
        response = classify_sentences(openai_client,
                                      str(conversation),
                                      prompt_system,
                                      prompt_user,
                                      prompt_agent)
        out.append(response)
    return out

# Definition of variables
**Should be modified for each run**

In [6]:
CODING_SCHEME = 'lin_dialogue_acts'  # 'lin_dialogue_acts' | 'acosta_epistemic_stance' | 'del_bosque_cma'
OUT_FILE = CODING_SCHEME + '.csv'
OUT_FOLDER = 'labeled_conversations'
LABEL_COLUMN = 'label'
CATEGORY_COLUMN = 'type'

## Classification

One activity at a time: `aux` lists the four activities, and the index picks the one to classify.

In [7]:
aux = list(df['activity_id'].unique())
aux

['asst_ncmAPAoB3MT4TIBrhhRAjpJ9',
 'asst_ZZ2f7t77l5PKcfG6pHkVMQDz',
 'asst_xRGU7Xk0G1eHkruFzY1kpaTO',
 'asst_Get3WE5ozTjkHcdGhnqGGqgL']

In [8]:
ACTIVITY = 0  # index in aux, from 0 to 3

act_1 = []
for x in conversations_json:
    if x['activity_id'] == aux[ACTIVITY]:
        act_1.append(x)

len(act_1)

23

In [8]:
from dotenv import load_dotenv

if RUN_API:
    # Load the API KEY
    load_dotenv(Path('..') / '.env')

    OPENAI_KEY = os.getenv("OPENAI_API_KEY")

    openai_client = OpenAI(api_key=OPENAI_KEY)

    prompt_system, prompt_user, prompt_agent = read_prompts(CODING_SCHEME)

    labeled_data = classify_conversation(openai_client,
                                         prompt_system,
                                         prompt_user,
                                         prompt_agent,
                                         act_1)

    errors

## Convert utterances to dataframe

The response comes back as JSON. The Dialogue act and the Conversational Management schemes return the
conversation with an `annotation` per sentence; the Epistemic Stance prompt returns messages holding
annotated `utterances`, so both shapes are handled here. Responses that are not strict JSON (single
quotes, mostly) are recovered with `demjson3`, as in the original runs.

In [9]:
import json
import demjson3


def convert_to_dataframe(list_of_json: list,
                         conversations: list,
                         label_column: str):
    flattened_data = []
    for entry, conversation in zip(list_of_json, conversations):
        try:
            if "'sentence_en'" in entry or '"sentence_en\'' in entry:
                entry = entry.replace("'sentence_en'", '"sentence_en"')
                entry = entry.replace('"sentence_en\'', '"sentence_en"')

            if '"role": \'model\'' in entry:
                entry = entry.replace('"role": \'model\'', '"role": "model"')

            json_entry = json.loads(entry)
        except:
            print(entry)
            json_entry = demjson3.decode(entry)

        activity_id = conversation['activity_id']
        user_id = conversation['user_id']
        role_of_message = {m['message_num']: m['role'] for m in conversation['conversation']}

        if 'conversation' in json_entry:
            for message in json_entry['conversation']:
                flattened_data.append({
                    'activity_id': activity_id,
                    'user_id': user_id,
                    'message_num': int(message['message_num']),
                    'role': message['role'],
                    'sentence_en': message['sentence_en'],
                    label_column: message['annotation']
                })
        else:
            # Epistemic Stance: {"annotations": [{"message_num": .., "utterances": [{"text": .., "annotation": ..}]}]}
            for message in json_entry['annotations']:
                message_num = int(message['message_num'])
                for utterance in message['utterances']:
                    flattened_data.append({
                        'activity_id': activity_id,
                        'user_id': user_id,
                        'message_num': message_num,
                        'role': role_of_message[message_num],
                        'sentence_en': utterance['text'],
                        label_column: utterance['annotation']
                    })

    out_df = pd.DataFrame(flattened_data)
    out_df['utterance_num'] = out_df.groupby(['user_id', 'message_num', 'role']).cumcount()
    out_df = out_df[['activity_id', 'user_id', 'role',
                     'message_num', 'utterance_num', 'sentence_en', label_column]]
    return out_df


if RUN_API:
    out_file = convert_to_dataframe(labeled_data, act_1, LABEL_COLUMN)
    out_file.head()

# Check that the dataframe has no labels with errors

When the model returns a label that is not in the scheme, the three researchers decided whether it
aligned with an existing category. Those decisions were kept as commented-out `str.replace` lines in
the original notebook; here they are the two dictionaries below, so that the step runs and can be
audited. `SPELLING_VARIANTS` holds variants of a valid label (the CMA example prompt spells
"Maintaining" while `tags.json` spells "Mantaining"), `LABEL_REMAP` the labels replaced by the closest
valid one. Whatever is still out of the scheme is set to `Other` and printed.

In [10]:
SPELLING_VARIANTS = {
    'del_bosque_cma': {
        'Tutor Floor Initiating or Floor Maintaining': 'Tutor Floor Initiating or Floor Mantaining',
        'Tutor Floor Initiating or Floor Mantining': 'Tutor Floor Initiating or Floor Mantaining',
    },
    'acosta_epistemic_stance': {
        'Other acts that maintaint discussion': 'Other acts that maintain discussion',
    },
}

LABEL_REMAP = {
    'lin_dialogue_acts': {
        'Acknowledgement': 'Acknowledge',
        'Elaboration': 'Information',
    },
    'acosta_epistemic_stance': {
        'Hedges or requests': 'Utterances that are hedges or requests',
    },
    'del_bosque_cma': {
        'Tutor Acknowledge Backchannel': 'Acknowledge Backchannel',
        'Tutor Floor Maintaining': 'Tutor Floor Initiating or Floor Mantaining',
    },
}

OTHER = 'Other'


def check_labels(out_file, coding_scheme, label_column=LABEL_COLUMN, category_column=CATEGORY_COLUMN):
    # Read coding scheme
    path = Path('..') / 'coding_schemes' / coding_scheme
    with open(path / 'tags.json') as f:
        tags = json.load(f)

    reversed_coding_scheme = {label: category for category, labels in tags.items() for label in labels}

    labels = out_file[label_column].astype(str).str.strip()
    labels = labels.replace(SPELLING_VARIANTS.get(coding_scheme, {}))

    remap = LABEL_REMAP.get(coding_scheme, {})
    out_of_scheme = ~labels.isin(reversed_coding_scheme)
    remapped = out_of_scheme & labels.isin(remap)
    to_other = out_of_scheme & ~labels.isin(remap)

    print('Checking labels')
    if not out_of_scheme.any():
        print('No conflicts')
    print(remapped.sum(), 'labels remapped,', to_other.sum(), 'assigned to', repr(OTHER))
    if to_other.any():
        print(labels[to_other].value_counts().to_dict())

    out_file = out_file.copy()
    out_file[label_column] = labels.where(~remapped, labels.map(remap)).where(~to_other, OTHER)
    out_file[category_column] = out_file[label_column].map(reversed_coding_scheme).fillna(OTHER)
    return out_file

# Save dataframe

In the original notebook each activity was saved on its own (`act_1lin_dialogue_acts.csv`, …) into
`labeled_conversations/aux_<scheme>/`, and the four files were concatenated afterwards in
`02_checks.ipynb`. Here the file of the activity is written to the same folder, and the cell below
concatenates whatever is there.

In [11]:
if RUN_API:
    out_file = check_labels(out_file, CODING_SCHEME)

    path = DATA / OUT_FOLDER / ('aux_' + CODING_SCHEME)
    path.mkdir(parents=True, exist_ok=True)

    out_file.to_csv(path / ('act_' + str(ACTIVITY) + CODING_SCHEME + '.csv'), index=False)

## Classify the whole corpus

The same steps for the three coding schemes and the four activities at once. With `RUN_API = False` it
reads the labels obtained in the study (`source/llm_labels_<scheme>.csv`) and runs only the checks,
which is what produces the files used by the rest of the pipeline.

In [12]:
CODING_SCHEMES = ['lin_dialogue_acts', 'acosta_epistemic_stance', 'del_bosque_cma']

path_out = DATA / OUT_FOLDER
path_out.mkdir(parents=True, exist_ok=True)

for coding_scheme in CODING_SCHEMES:

    if RUN_API:
        prompt_system, prompt_user, prompt_agent = read_prompts(coding_scheme)
        labeled_data = classify_conversation(openai_client,
                                             prompt_system,
                                             prompt_user,
                                             prompt_agent,
                                             conversations_json)
        out_file = convert_to_dataframe(labeled_data, conversations_json, LABEL_COLUMN)
    else:
        out_file = pd.read_csv(DATA / 'source' / ('llm_labels_' + coding_scheme + '.csv'))

    # Keep the order of the corpus (notebook 00) and drop the conversations that are not in it
    order = df[['activity_id', 'user_id']].drop_duplicates().reset_index(drop=True)
    order['conversation_order'] = order.index
    out_file = out_file.merge(order, on=['activity_id', 'user_id'])
    out_file = out_file.sort_values(['conversation_order', 'message_num', 'utterance_num'],
                                    kind='stable').drop(columns='conversation_order')

    out_file = check_labels(out_file, coding_scheme)
    out_file.to_csv(path_out / (coding_scheme + '.csv'), index=False)

out_file.head()

Checking labels
0 labels remapped, 3 assigned to 'Other'
{'Other': 3}
Checking labels
No conflicts
0 labels remapped, 0 assigned to 'Other'
Checking labels
0 labels remapped, 4 assigned to 'Other'
{'Student Statement': 4}


,activity_id,user_id,role,message_num,utterance_num,sentence_en,label,type
0,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,model,0,0,"Hello! 😸 I am SIMBA, and I will help you think...",Tutor Floor Initiating or Floor Mantaining,Tutor Floor Initiating or Floor Mantaining
1,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,model,0,1,What are the 2 to 5 main ideas to remember fro...,Tutor Elicit Response,Tutor Elicit Response
2,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,model,0,2,What are the points of the course that still s...,Tutor Elicit Response,Tutor Elicit Response
3,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,model,0,3,I am here to accompany you in this reflection! 🌟,Tutor Floor Initiating or Floor Mantaining,Tutor Floor Initiating or Floor Mantaining
4,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,user,1,0,Can you clarify the difference between an isol...,Student Question,Student Question


In [13]:
for coding_scheme in CODING_SCHEMES:
    labeled = pd.read_csv(path_out / (coding_scheme + '.csv'))
    print(coding_scheme, ':', len(labeled), 'utterances,',
          labeled['label'].nunique(), 'labels,', labeled['type'].nunique(), 'first-level categories')

lin_dialogue_acts : 4877 utterances, 31 labels, 13 first-level categories
acosta_epistemic_stance : 4928 utterances, 5 labels, 5 first-level categories
del_bosque_cma : 4963 utterances, 11 labels, 11 first-level categories
